# COMP5329 — Deep Learning

**Tutorial 9 — Multi-Modal Foundation Models: From Language to Vision to Action**

**Semester 1, 2026**

### Learning Objectives

By the end of this tutorial you will be able to:

1. Derive **cross-attention** from self-attention and implement it from scratch, explaining when Q, K, V come from different sources.
2. Explain the **KV cache** optimisation for autoregressive generation and implement it from scratch, demonstrating the computational savings.
3. Describe the full **LLM training pipeline**: pre-training, supervised fine-tuning (SFT), and alignment (RLHF/DPO), explaining how each stage transforms model behaviour.
4. Explain the **reward model** in RLHF and derive the **DPO objective** as a closed-form alternative to PPO-based alignment.
5. Describe how **CLIP** uses contrastive learning to align vision and language representations in a shared embedding space.
6. Explain how **LLaVA** connects a frozen vision encoder to an LLM via a learned projection layer.
7. Explain **Flamingo's** gated cross-attention mechanism for interleaving visual and textual information.
8. Compare the three VLM paradigms (contrastive, projective, cross-attention) on axes of architecture, training, and capability.
9. Describe how **Vision-Language-Action (VLA)** models extend VLMs to produce physical actions, using RT-2 and OpenVLA as examples.
10. Evaluate the progression from text-only to multi-modal to embodied foundation models on a unified comparison table.

### Topic Coverage

Week 9 covers **multi-modal foundation models: from language to vision to action**. The full topic list (see `Week9_Self_Study_Material.ipynb`) is:

- ✅ **Large language models** — pre-training, SFT, RLHF, DPO alignment *(tutorial)*
- ✅ **Cross-attention** — fusing two information sources, from-scratch implementation *(tutorial, with in-class practice)*
- ✅ **KV cache** — autoregressive generation speedup, prefill vs decode *(tutorial, with in-class practice)*
- ✅ **Vision-Language Models** — CLIP (contrastive), LLaVA (projective), Flamingo (cross-attention) *(tutorial, with in-class practice on CLIP loss)*
- 📖 **LoRA / parameter-efficient fine-tuning** — low-rank adaptation for LLMs *(self-study)*
- 📖 **Mixture of Experts (MoE)** — sparse routing, scaling laws with experts *(self-study)*
- ✅ **Vision-Language-Action (VLA)** — RT-2, OpenVLA, actions as tokens *(tutorial, conceptual)*
- 📖 **Evolution of foundation models across Weeks 7-9** — unified view *(self-study)*

Due to time constraints, the live tutorial focuses on the **three fundamental mechanisms** (cross-attention, KV cache, RLHF/DPO) and the **three VLM paradigms**. LoRA, MoE, and deep coverage of VLA are left as self-study — the self-study notebook covers these in depth.

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding exercises, **Part C** — exam-style Q&A.

---
# Part A · Tutor Walkthrough

## 0. The Story So Far

Last week we studied GPT (decoder-only, pre-training, scaling laws, in-context learning) and BERT (encoder-only, masked language model). This week we ask **three questions**:

1. **How does GPT become ChatGPT?** — instruction tuning and alignment with human preferences.
2. **What if models need to see AND talk?** — cross-attention as the bridge between modalities, and three VLM paradigms (CLIP, LLaVA, Flamingo).
3. **What if models need to act in the physical world?** — from VLM to VLA: Vision-Language-Action models for robotics.

We start with two **fundamental mechanisms** (cross-attention, KV cache) that underpin everything else, then build up to multi-modal and embodied systems.

---

## 1. Cross-Attention: Fusing Two Information Sources

### 1.1 From Self-Attention to Cross-Attention

In Week 7 we built **self-attention**, where Query, Key, and Value all come from the same sequence $X$:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$
$$\text{SelfAttn}(X) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

But what if we want one sequence to **query information from another**? For example: a text decoder querying visual features, or a translation model querying the source language.

**Cross-attention** makes one simple change: Q comes from one source, while K and V come from a different source:

$$Q = X_{\text{query}}\, W^Q, \quad K = X_{\text{context}}\, W^K, \quad V = X_{\text{context}}\, W^V$$
$$\text{CrossAttn}(X_{\text{query}}, X_{\text{context}}) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

If $X_{\text{query}} \in \mathbb{R}^{T_q \times d}$ and $X_{\text{context}} \in \mathbb{R}^{T_c \times d}$, the attention weight matrix is $\mathbb{R}^{T_q \times T_c}$ — **not square**! Each query token attends over all context tokens, and the output has the same length as the query ($T_q$).

### 1.2 From-Scratch Implementation

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────────
import math
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline

In [ ]:
# ── Cross-Attention ──────────────────────────────────────────────────────────

class CrossAttention(nn.Module):
    """Multi-head cross-attention: Q from query sequence, K/V from context sequence."""
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)  # query projection (from query seq)
        self.W_k = nn.Linear(d_model, d_model)  # key projection (from context seq)
        self.W_v = nn.Linear(d_model, d_model)  # value projection (from context seq)
        self.W_o = nn.Linear(d_model, d_model)  # output projection

    def forward(self, x_query, x_context):
        """
        Args:
            x_query:   (B, T_q, d_model) -- the sequence asking questions
            x_context: (B, T_c, d_model) -- the sequence providing answers
        Returns:
            output: (B, T_q, d_model), attn_weights: (B, num_heads, T_q, T_c)
        """
        B, T_q, _ = x_query.size()
        _, T_c, _ = x_context.size()

        # Project: Q from query, K/V from context
        Q = self.W_q(x_query).view(B, T_q, self.num_heads, self.d_k).transpose(1, 2)   # (B, H, T_q, d_k)
        K = self.W_k(x_context).view(B, T_c, self.num_heads, self.d_k).transpose(1, 2) # (B, H, T_c, d_k)
        V = self.W_v(x_context).view(B, T_c, self.num_heads, self.d_k).transpose(1, 2) # (B, H, T_c, d_k)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, H, T_q, T_c)
        attn_weights = F.softmax(scores, dim=-1)                                # (B, H, T_q, T_c)
        out = torch.matmul(attn_weights, V)                                     # (B, H, T_q, d_k)

        # Concatenate heads and project
        out = out.transpose(1, 2).contiguous().view(B, T_q, -1)  # (B, T_q, d_model)
        return self.W_o(out), attn_weights

In [ ]:
# ── Verification ─────────────────────────────────────────────────────────────
torch.manual_seed(42)
d_model, num_heads = 64, 4
cross_attn = CrossAttention(d_model, num_heads)

# Text query (5 tokens) attending to image context (16 patches)
x_query = torch.randn(2, 5, d_model)    # (B=2, T_q=5, d=64)
x_context = torch.randn(2, 16, d_model)  # (B=2, T_c=16, d=64)

out, attn = cross_attn(x_query, x_context)
print(f'Query shape:     {x_query.shape}   (B, T_q, d)')
print(f'Context shape:   {x_context.shape}  (B, T_c, d)')
print(f'Output shape:    {out.shape}   (B, T_q, d)  -- same length as query!')
print(f'Attention shape: {attn.shape}  (B, heads, T_q, T_c)  -- rectangular!')
print(f'Row sums: {attn[0, 0].sum(dim=-1).detach().numpy().round(6)}  -- each row sums to 1')

### 1.3 Visualising Cross-Attention

In [ ]:
# ── Cross-attention heatmap ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3))
weights = attn[0, 0].detach().numpy()  # head 0, batch 0: (T_q=5, T_c=16)
im = ax.imshow(weights, aspect='auto', cmap='viridis')
ax.set_xlabel('Context tokens (image patches)', fontsize=11)
ax.set_ylabel('Query tokens (text)', fontsize=11)
ax.set_yticks(range(5))
ax.set_yticklabels([f'text_{i}' for i in range(5)])
ax.set_title('Cross-Attention Weights: Text queries attending to Image patches', fontsize=12)
plt.colorbar(im, ax=ax, label='Attention weight')
plt.tight_layout()
plt.show()
print('Each row shows how much one text token attends to each image patch.')

### 1.4 Self-Attention vs. Cross-Attention

| Aspect | Self-Attention | Cross-Attention |
|---|---|---|
| Q source | Same sequence $X$ | Query sequence $X_q$ |
| K, V source | Same sequence $X$ | Context sequence $X_c$ |
| Weight matrix shape | $T \times T$ (square) | $T_q \times T_c$ (rectangular) |
| Use case | Relate tokens within a sequence | Fuse information across sources |
| Examples | BERT, GPT layers | Encoder-decoder, VLM fusion |

> **Transition**: Cross-attention lets one sequence query another -- this is exactly how VLMs will connect vision and language (Section 4). But first, there is an efficiency problem in autoregressive generation. Each time we generate a new token, we recompute K and V for ALL previous tokens. Can we avoid this redundant computation?

---

## 2. KV Cache: Making Autoregressive Generation Fast

### 2.1 The Problem

In autoregressive generation (GPT, Week 8), at step $t$ we generate token $x_t$ conditioned on $x_1, \ldots, x_{t-1}$. A naive implementation **recomputes** the full attention for the entire sequence at every step.

At step $t=2$: we compute $Q_1, K_1, V_1, Q_2, K_2, V_2$ from scratch. But $K_1$ and $V_1$ are **identical** to what we computed at step $t=1$ -- wasted work!

### 2.2 How KV Cache Works

**Without cache** (step $t$) -- recompute everything:

$$Q_{1:t} = X_{1:t} W^Q, \quad K_{1:t} = X_{1:t} W^K, \quad V_{1:t} = X_{1:t} W^V$$

**With cache** (step $t$) -- only compute for the new token, reuse cached K, V:

$$q_t = x_t W^Q, \quad k_t = x_t W^K, \quad v_t = x_t W^V$$
$$K_{1:t} = \text{concat}(K_{\text{cache}}, k_t), \quad V_{1:t} = \text{concat}(V_{\text{cache}}, v_t)$$
$$\text{out}_t = \text{softmax}\!\left(\frac{q_t\, K_{1:t}^\top}{\sqrt{d_k}}\right) V_{1:t}$$

The cache stores all previously computed keys and values. At each step, we only project the **single new token** and append to the cache.

### 2.3 From-Scratch Implementation

In [ ]:
# ── Naive causal attention (no cache) ─────────────────────────────────────

class NaiveCausalAttention(nn.Module):
    """Recomputes full attention at every generation step."""
    def __init__(self, d_model):
        super().__init__()
        self.d = d_model
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        """x: (B, T, d) -- full sequence. Returns last position output."""
        B, T, d = x.shape
        Q = self.W_q(x)  # (B, T, d)
        K = self.W_k(x)  # (B, T, d)
        V = self.W_v(x)  # (B, T, d)
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(d)  # (B, T, T)
        mask = torch.tril(torch.ones(T, T, device=x.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)  # (B, T, d)
        return out[:, -1:]  # only need the last position's output

In [ ]:
# ── Cached causal attention ────────────────────────────────────────────────

class CachedCausalAttention(nn.Module):
    """Only computes Q/K/V for the new token; caches K, V from previous steps."""
    def __init__(self, d_model):
        super().__init__()
        self.d = d_model
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x_new, kv_cache=None):
        """
        Args:
            x_new: (B, 1, d) -- embedding of the new token only
            kv_cache: tuple (K_cache, V_cache) each (B, t-1, d), or None for first token
        Returns:
            output: (B, 1, d), updated_cache: (K, V) each (B, t, d)
        """
        q = self.W_q(x_new)      # (B, 1, d)
        k_new = self.W_k(x_new)  # (B, 1, d)
        v_new = self.W_v(x_new)  # (B, 1, d)

        if kv_cache is not None:
            K = torch.cat([kv_cache[0], k_new], dim=1)  # (B, t, d)
            V = torch.cat([kv_cache[1], v_new], dim=1)  # (B, t, d)
        else:
            K, V = k_new, v_new

        # Attention: q attends to ALL cached keys (no mask needed -- all are past tokens)
        scores = torch.matmul(q, K.transpose(-1, -2)) / math.sqrt(self.d)  # (B, 1, t)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)  # (B, 1, d)
        return out, (K, V)

In [ ]:
# ⭐ Extension: Timing benchmark (run as take-home exercise)
# ── Timing comparison ───────────────────────────────────────────────────────
torch.manual_seed(42)
d_model = 64

# Share the same weights for fair comparison
naive_attn = NaiveCausalAttention(d_model)
cached_attn = CachedCausalAttention(d_model)
cached_attn.W_q.weight = naive_attn.W_q.weight
cached_attn.W_k.weight = naive_attn.W_k.weight
cached_attn.W_v.weight = naive_attn.W_v.weight

seq_lengths = [32, 64, 128, 256, 512]
naive_times = []
cached_times = []

for T in seq_lengths:
    embeddings = torch.randn(1, T, d_model)  # pre-generated token embeddings

    # Naive: at each step, pass the full sequence so far
    torch.manual_seed(0)
    t0 = time.time()
    with torch.no_grad():
        for t in range(1, T + 1):
            _ = naive_attn(embeddings[:, :t])
    naive_times.append(time.time() - t0)

    # Cached: at each step, pass only the new token + cache
    torch.manual_seed(0)
    t0 = time.time()
    with torch.no_grad():
        cache = None
        for t in range(T):
            _, cache = cached_attn(embeddings[:, t:t+1], cache)
    cached_times.append(time.time() - t0)

    print(f'T={T:>4d}  naive: {naive_times[-1]:.3f}s  cached: {cached_times[-1]:.3f}s  '
          f'speedup: {naive_times[-1]/cached_times[-1]:.1f}x')

In [ ]:
# ── Speedup visualisation ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(seq_lengths, naive_times, 'o-', label='Naive (no cache)', color='#e74c3c')
ax1.plot(seq_lengths, cached_times, 's-', label='With KV Cache', color='#2ecc71')
ax1.set_xlabel('Sequence Length T')
ax1.set_ylabel('Wall-clock Time (s)')
ax1.set_title('Generation Time: Naive vs KV Cache')
ax1.legend()
ax1.grid(True, alpha=0.3)

speedups = [n / c for n, c in zip(naive_times, cached_times)]
ax2.bar(range(len(seq_lengths)), speedups, tick_label=[str(s) for s in seq_lengths],
        color='#3498db')
ax2.set_xlabel('Sequence Length T')
ax2.set_ylabel('Speedup Factor')
ax2.set_title('KV Cache Speedup')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 2.5 The Memory-Compute Trade-off

KV cache trades **memory** for **compute**. For a model with $L$ layers, $H$ heads, head dimension $d_k$, generating a sequence of length $T$:

$$\text{Cache size} = 2 \times L \times H \times T \times d_k \quad (\text{factor 2 for K and V})$$

For a 7B parameter model (32 layers, 32 heads, $d_k = 128$), generating 4096 tokens:
$$2 \times 32 \times 32 \times 4096 \times 128 \times 2 \text{ bytes (fp16)} \approx 1\text{ GB}$$

This is why long-context LLMs need substantial GPU memory at inference time.

| | Without Cache | With Cache |
|---|---|---|
| Per-step compute | $O(t^2 \cdot d)$ | $O(t \cdot d)$ |
| Per-step extra memory | $O(1)$ | $O(t \cdot d)$ cached |
| Total compute ($T$ steps) | $O(T^3 \cdot d)$ | $O(T^2 \cdot d)$ |
| Latency per token | Grows quadratically | Grows linearly |

### 2.6 Prefill vs. Decode Phases

Modern LLM serving has two distinct phases:

1. **Prefill**: Process the entire user prompt **in parallel** (like standard Transformer forward pass). Populate the KV cache. This is why the first token takes longer.
2. **Decode**: Generate tokens **one at a time** using the cache. Each subsequent token is fast because we only process one new token.

This two-phase structure is fundamental to understanding LLM inference latency.

> **Transition**: We now understand two key mechanisms: cross-attention (fusing information across modalities) and KV cache (efficient autoregressive generation). With these tools in hand, let us answer the big question: how does a base GPT model become a useful assistant like ChatGPT?

---

## 3. The LLM Training Pipeline: From GPT to ChatGPT

### 3.1 Three-Stage Overview

```
Stage 1: Pre-training        Stage 2: SFT              Stage 3: Alignment
(Week 8)                     (This section)             (This section)
┌───────────────┐         ┌──────────────────┐      ┌─────────────────────┐
│ Next-token      │   →    │ Instruction      │  →   │ RLHF or DPO        │
│ prediction on   │         │ following on     │       │ (human preferences) │
│ web text        │         │ (prompt, resp.)  │       │                     │
└───────────────┘         └──────────────────┘      └─────────────────────┘
 "Predict next"          "Follow instructions"     "Be helpful & safe"
```

### 3.2 Stage 1: Pre-training (Review)

Covered in Week 8 Section 1. The causal language model objective on massive corpora (trillions of tokens):

$$\mathcal{L}_{\text{pretrain}} = -\sum_{t=1}^{T} \log P(x_t \mid x_{1:t-1}; \theta)$$

The result is a **base model** that can complete text but does not follow instructions. Given "What is the capital of France?", it might continue with "What is the capital of Germany? What is..." rather than answering "Paris."

### 3.3 Stage 2: Supervised Fine-Tuning (SFT)

Collect a dataset of **(instruction, response)** pairs -- human-written examples of good responses. Fine-tune the base model using the same causal LM objective, but **only on the response tokens**:

$$\mathcal{L}_{\text{SFT}} = -\sum_{t=|\text{prompt}|+1}^{T} \log P(x_t \mid x_{1:t-1}; \theta)$$

The loss is **masked on the prompt tokens** -- we train the model to generate good responses, not to predict prompts.

After SFT: the model follows instructions, but may still produce harmful, biased, or untruthful outputs.

### 3.4 Stage 3: Alignment with Human Preferences

**RLHF** (Reinforcement Learning from Human Feedback) pipeline:

1. **Collect comparisons**: For a prompt, generate multiple responses. Humans rank them: $y_w \succ y_l$ (preferred vs. dispreferred).
2. **Train a reward model** $r_\phi(x, y)$: Predicts a scalar score for (prompt, response) pairs using the Bradley-Terry preference model:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l)} \left[\log \sigma\bigl(r_\phi(x, y_w) - r_\phi(x, y_l)\bigr)\right]$$

3. **Optimise via PPO**: Fine-tune the LLM to maximise the reward while staying close to the SFT model (KL penalty):

$$\max_\theta \; \mathbb{E}_{x,\, y \sim \pi_\theta} \left[r_\phi(x, y)\right] - \beta \, D_{\text{KL}}\left[\pi_\theta \| \pi_{\text{SFT}}\right]$$

The KL term prevents **reward hacking** -- the model exploiting the reward model's blind spots.

### 3.5 DPO: Skipping the Reward Model

**Key insight** (Rafailov et al., 2023): The optimal policy under the RLHF objective has a **closed-form solution**. We can reparametrise the reward in terms of the policy itself, giving the **DPO loss** -- directly on the language model, no separate reward model needed:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}_{(x, y_w, y_l)} \left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right]$$

where $\pi_{\text{ref}}$ is the frozen SFT model.

**Why DPO is elegant**: No reward model to train, no PPO instability, no online generation during training. Just a classification-like loss on preference pairs.

In [ ]:
# ── DPO loss implementation ─────────────────────────────────────────────────

def dpo_loss(pi_logprobs_w, pi_logprobs_l, ref_logprobs_w, ref_logprobs_l, beta=0.1):
    """Compute DPO loss for a batch of preference pairs.
    Args:
        pi_logprobs_w:  log pi_theta(y_w | x) for preferred responses   (B,)
        pi_logprobs_l:  log pi_theta(y_l | x) for dispreferred responses (B,)
        ref_logprobs_w: log pi_ref(y_w | x)                              (B,)
        ref_logprobs_l: log pi_ref(y_l | x)                              (B,)
        beta: temperature parameter
    Returns:
        Scalar DPO loss.
    """
    log_ratio_w = pi_logprobs_w - ref_logprobs_w  # (B,)
    log_ratio_l = pi_logprobs_l - ref_logprobs_l  # (B,)
    logits = beta * (log_ratio_w - log_ratio_l)    # (B,)
    return -F.logsigmoid(logits).mean()

In [ ]:
# ── DPO verification ───────────────────────────────────────────────────────
# Scenario: policy correctly prefers the preferred response
pi_w  = torch.tensor([-1.0, -1.5])   # log P(y_w) under policy
pi_l  = torch.tensor([-3.0, -4.0])   # log P(y_l) under policy (lower = less likely)
ref_w = torch.tensor([-2.0, -2.0])   # log P(y_w) under reference
ref_l = torch.tensor([-2.0, -2.0])   # log P(y_l) under reference

loss_correct = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1)
print(f'Loss when policy CORRECTLY prefers y_w: {loss_correct.item():.4f}')

# Scenario: policy incorrectly prefers the dispreferred response
loss_wrong = dpo_loss(pi_l, pi_w, ref_w, ref_l, beta=0.1)  # swap preferred/dispreferred
print(f'Loss when policy INCORRECTLY prefers y_l: {loss_wrong.item():.4f}')
print(f'\nCorrect preference gives LOWER loss, as expected.')

### 3.7 LLM Pipeline Summary

| Stage | Objective | Data | Result |
|---|---|---|---|
| **Pre-training** | Next-token prediction | Trillions of tokens (web) | Base model (text completion) |
| **SFT** | Next-token on responses | ~100K (instruction, response) pairs | Instruction-following model |
| **RLHF** | Maximise reward $-$ KL | Human preference rankings | Aligned model (helpful + safe) |
| **DPO** | Direct preference optimisation | Same preference data | Aligned model (simpler pipeline) |

**Open vs. closed models**: GPT-4, Claude, Gemini are closed-source. LLaMA, Mistral, Qwen are open-weight. The training pipeline is the same; the difference is access.

> **Transition**: We now understand how LLMs go from raw text prediction to helpful assistants. But language is just **one modality**. The real world is visual, auditory, and spatial. How do we extend these powerful language models to ALSO understand images? This requires connecting a vision encoder to an LLM -- and the key mechanisms are **cross-attention** (Section 1) and **visual projection**.

---

## 4. Vision-Language Models: Teaching LLMs to See

### 4.0 The Challenge

Images and text live in **different representation spaces**. Images are grids of pixels; text is sequences of discrete tokens. How do we bridge this gap?

Three paradigms, in order of increasing integration depth:

1. **CLIP** -- Align vision and language in a shared space (contrastive learning, **no generation**)
2. **LLaVA** -- Project vision features into the LLM's token space (simple fusion, **full generation**)
3. **Flamingo** -- Interleave visual information via gated cross-attention (deep fusion, **few-shot capable**)

### 4.1 CLIP: Contrastive Language-Image Pre-training

**Architecture**: Two separate encoders -- a **Vision Transformer (ViT)** for images and a **Transformer** for text -- trained to align matching (image, text) pairs in a shared embedding space.

**ViT brief** (connecting Week 5 CNNs to Transformers): Split an image into non-overlapping patches (e.g., 16$\times$16 pixels). Linearly project each patch into an embedding. Process the sequence of patch embeddings with a standard Transformer encoder. This is "CNN meets Transformer."

**Contrastive objective** (InfoNCE): In a batch of $N$ (image, text) pairs, maximise similarity for **matching** pairs and minimise it for **mismatched** pairs:

$$\mathcal{L}_{\text{CLIP}} = -\frac{1}{N}\sum_{i=1}^{N}\left[\log \frac{e^{\text{sim}(v_i, t_i)/\tau}}{\sum_{j=1}^{N} e^{\text{sim}(v_i, t_j)/\tau}} + \log \frac{e^{\text{sim}(t_i, v_i)/\tau}}{\sum_{j=1}^{N} e^{\text{sim}(t_i, v_j)/\tau}}\right]$$

where $\text{sim}(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$ is cosine similarity and $\tau$ is a learned temperature.

**CLIP can**: zero-shot image classification (compare image embedding to text embeddings of class names), image-text retrieval.

**CLIP cannot**: generate text, answer detailed questions about images, have conversations.

In [ ]:
# ── CLIP contrastive loss ────────────────────────────────────────────────────

def clip_loss(image_embeds, text_embeds, temperature=0.07):
    """Compute CLIP symmetric contrastive loss.
    Args:
        image_embeds: (N, D) L2-normalised image embeddings
        text_embeds:  (N, D) L2-normalised text embeddings
        temperature:  learned temperature parameter
    Returns:
        Scalar loss.
    """
    logits = torch.matmul(image_embeds, text_embeds.T) / temperature  # (N, N)
    labels = torch.arange(len(image_embeds), device=image_embeds.device)
    loss_i2t = F.cross_entropy(logits, labels)      # image-to-text
    loss_t2i = F.cross_entropy(logits.T, labels)    # text-to-image
    return (loss_i2t + loss_t2i) / 2

In [ ]:
# ── CLIP similarity matrix visualisation ─────────────────────────────────
torch.manual_seed(42)
N, D = 6, 128

# Simulate: create embeddings where matching pairs are similar
shared = torch.randn(N, D)
image_emb = F.normalize(shared + torch.randn(N, D) * 0.3, dim=-1)
text_emb  = F.normalize(shared + torch.randn(N, D) * 0.3, dim=-1)

sim_matrix = (image_emb @ text_emb.T).detach().numpy()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-0.5, vmax=1.0)
ax.set_xlabel('Text embeddings')
ax.set_ylabel('Image embeddings')
ax.set_title('CLIP Similarity Matrix (diagonal = matching pairs)')
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

loss = clip_loss(image_emb, text_emb)
print(f'CLIP loss: {loss.item():.4f}  (lower = better alignment on diagonal)')

### 4.2 LLaVA: Visual Instruction Tuning

**Architecture**: Frozen CLIP ViT (vision encoder) $\to$ Learned projection layer (MLP) $\to$ Frozen/fine-tuned LLM (e.g., LLaMA).

```
Image → [ViT Encoder] → visual features (N_patches, D_vision)
                              ↓
                    [Projection MLP] → visual tokens (N_patches, D_llm)
                              ↓
                    Concatenate with text tokens
                              ↓
              [LLM] → generate text response
```

**Key insight**: The projection MLP learns to map visual features into the LLM's token embedding space. After projection, the LLM treats visual features **just like text tokens** -- no architectural changes to the LLM needed!

**Training (2 stages)**:
1. **Pre-training**: Train only the projection MLP on image-caption pairs (align visual features to text space).
2. **Fine-tuning**: Fine-tune projection + LLM on visual instruction data (QA pairs about images).

### 4.3 Flamingo: Gated Cross-Attention for Interleaved Vision-Language

**Architecture**: Frozen vision encoder $+$ **Perceiver Resampler** $+$ **Gated cross-attention** layers inserted into a frozen LLM.

**Perceiver Resampler**: Reduces the variable number of visual features to a **fixed number** of visual tokens (e.g., 64) using learned queries and cross-attention. This controls compute cost regardless of image resolution.

**Gated cross-attention**: Inserted between existing LLM layers. The LLM's hidden states (Q) attend to visual tokens (K, V). A learned gate $\tanh(\alpha)$ starts at 0, so the model initially behaves exactly like the original LLM:

$$\text{out} = x_{\text{text}} + \tanh(\alpha) \cdot \text{CrossAttn}(x_{\text{text}},\; x_{\text{visual}})$$

Why $\tanh(\alpha)$ with $\alpha$ initialised to 0? Because $\tanh(0) = 0$, the visual information is initially **completely blocked**. The pre-trained LLM weights are perfectly preserved at initialisation. During training, $\alpha$ gradually increases, "mixing in" visual information.

This is where **cross-attention from Section 1** directly applies!

In [ ]:
# ── Gated Cross-Attention Layer (Flamingo-style) ───────────────────────

class GatedCrossAttentionLayer(nn.Module):
    """Flamingo-style gated cross-attention that preserves LLM behaviour at init."""
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.cross_attn = CrossAttention(d_model, num_heads)  # reuse from Section 1!
        self.gate = nn.Parameter(torch.zeros(1))  # tanh(0) = 0 → starts as identity
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x_text, x_visual):
        """
        Args:
            x_text:   (B, T_text, d) -- LLM hidden states
            x_visual: (B, T_vis, d)  -- visual tokens from Perceiver
        Returns:
            (B, T_text, d) -- text hidden states enriched with visual information
        """
        attn_out, _ = self.cross_attn(self.ln(x_text), x_visual)  # (B, T_text, d)
        return x_text + torch.tanh(self.gate) * attn_out           # gated residual


# ── Verification ──
torch.manual_seed(42)
d_model = 64
gated_layer = GatedCrossAttentionLayer(d_model, num_heads=4)
x_text = torch.randn(2, 10, d_model)   # 10 text tokens
x_visual = torch.randn(2, 64, d_model)  # 64 visual tokens from Perceiver
out = gated_layer(x_text, x_visual)
print(f'Input text shape:  {x_text.shape}')
print(f'Visual tokens:     {x_visual.shape}')
print(f'Output shape:      {out.shape}')
print(f'Gate value (init): tanh({gated_layer.gate.item():.4f}) = {torch.tanh(gated_layer.gate).item():.4f}')
print(f'Output == Input at init? {torch.allclose(out, x_text, atol=1e-6)}')
print('\nAt initialisation, the gated cross-attention is a no-op (gate = 0).')
print('The LLM behaves exactly as before. Visual information is mixed in during training.')

### 4.5 Three VLM Paradigms Compared

| Aspect | CLIP | LLaVA | Flamingo |
|---|---|---|---|
| **Fusion method** | Contrastive alignment | Linear/MLP projection | Gated cross-attention |
| **Vision encoder** | ViT (trained) | ViT (frozen, from CLIP) | NFNet (frozen) |
| **LLM** | None | LLaMA (fine-tuned) | Chinchilla (frozen) |
| **Can generate text?** | No | Yes | Yes |
| **Few-shot capable?** | Zero-shot only | No (needs fine-tuning) | Yes (interleaved images+text) |
| **Training cost** | High (from scratch) | Low (only projection + LLM) | Medium (cross-attn layers) |
| **Key strength** | Zero-shot transfer | Simple and effective | Flexible, few-shot |

### 4.6 VLM Design Space

Key architectural decisions when building a VLM:

1. **How to encode vision**: ViT, SigLIP, DINOv2, ...
2. **How to connect to LLM**: projection (LLaVA), cross-attention (Flamingo), Q-Former (BLIP-2)
3. **What to freeze**: Freezing vision encoder is standard; freezing vs. fine-tuning LLM is a cost-performance trade-off.
4. **Resolution handling**: More patches = more visual tokens = higher compute.

### 4.7 VLM Capabilities and Limitations

**Can do**: Visual question answering, image captioning, visual reasoning, OCR, chart understanding, document analysis.

**Still struggles with**: Spatial reasoning, counting, fine-grained visual grounding, hallucination (describing objects not in the image).

> **Transition**: VLMs can see and talk, but they cannot **act**. A VLM can describe a scene or answer questions about it, but it cannot pick up an object, navigate a room, or fold a shirt. The final frontier is connecting perception and language to **physical actions**.

---

## 5. Vision-Language-Action Models: From Seeing to Doing ⭐ *Extension*

> *VLA is cutting-edge research. Cover the core idea (actions as tokens) in 2 minutes; the detailed discussion is for self-study.*

### 5.1 The VLA Hypothesis

A single model that takes in **(image, language instruction)** and outputs **(actions)**. The key insight: tokenise actions the same way we tokenise text.

```
VLM:  (image, text) → text
VLA:  (image, text) → actions (discretised into tokens)
```

By representing actions as tokens, we can leverage the entire LLM infrastructure: pre-training, scaling, in-context learning, KV cache for fast inference.

### 5.2 RT-2: Robotic Transformer 2

**Google DeepMind (2023)**. Key ideas:

- Start with a VLM (PaLI-X or PaLM-E)
- Represent robot actions as **text tokens**: discretise each action dimension (x, y, z, rotation, gripper) into 256 bins, encode as token IDs
- Fine-tune the VLM on (image, instruction, action) tuples
- At inference: the model "speaks" actions just like it speaks text

**Example**: Given an image of a table with objects and the instruction "pick up the red cup", the model outputs action tokens that are decoded into motor commands.

**Why this works**: Language models are already good at sequential decision-making (next-token prediction IS a sequential decision). By tokenising actions, we leverage all the language model's capabilities.

### 5.3 OpenVLA: Open-Source VLA

**Architecture**: SigLIP (vision) + LLaMA (language) + action tokenisation.

Key contribution: open-source, reproducible VLA demonstrating that the VLM-to-VLA pipeline is general.

Training recipe: pre-train as a VLM, then fine-tune on robot manipulation data (Open X-Embodiment dataset -- a collection of robot experiences from many labs).

| Aspect | RT-2 | OpenVLA |
|---|---|---|
| Base model | PaLI-X / PaLM-E | SigLIP + LLaMA |
| Action representation | 256-bin discretisation | 256-bin discretisation |
| Training data | Google robot data | Open X-Embodiment |
| Open source | No | Yes |
| Key insight | Actions as tokens | VLM-to-VLA is general |

### 5.4 Challenges and Open Questions

1. **Action representation**: Discretisation loses precision. Continuous action spaces may need better tokenisation.
2. **Embodiment gap**: A model trained on one robot does not directly transfer to another (different joints, sensors, dynamics).
3. **Safety**: Incorrect actions have **physical consequences** -- unlike incorrect text, a wrong motor command can break things.
4. **Data scarcity**: Robot data is orders of magnitude scarcer than text/image data (~millions vs. trillions of tokens).
5. **Real-time inference**: Robots need fast responses; LLM inference can be slow (but KV cache from Section 2 helps!).

### 5.5 The Full Progression

```
Text only             →  Multi-modal          →  Embodied
(Week 8)                 (Week 9)                 (Week 9)

GPT / BERT               CLIP, LLaVA,             RT-2, OpenVLA
                         Flamingo

"understand text"        "see AND talk"           "see, talk, AND act"

Architecture:            Key mechanisms:           Key innovation:
Decoder / Encoder        Cross-attention,          Action
Transformer              Projection,               tokenisation
                         Contrastive learning
```

---

## 6. Grand Comparison

### 6.1 Unified Comparison Table

| Feature | Cross-Attention | KV Cache | LLM (ChatGPT) | CLIP | LLaVA | Flamingo | RT-2 / OpenVLA |
|---|---|---|---|---|---|---|---|
| **Type** | Mechanism | Optimisation | Training pipeline | VLM (contrastive) | VLM (projective) | VLM (cross-attn) | VLA |
| **Modalities** | Any two | Single (text) | Text only | Vision + Language | Vision + Language | Vision + Language | Vision + Language + Action |
| **Core innovation** | Q from one source, KV from another | Cache K,V across steps | SFT + RLHF/DPO | Contrastive alignment | Projection into LLM space | Gated cross-attention | Actions as tokens |
| **Can generate?** | (mechanism) | (optimisation) | Yes (text) | No | Yes (text) | Yes (text) | Yes (text + actions) |
| **Code depth** | From scratch | From scratch | DPO code | Loss code | Conceptual | Gated xattn code | Conceptual |

### 6.2 The Evolution of Foundation Models (Weeks 7-9)

```
Week 7:  RNN → LSTM → GRU → Transformer (self-attention)
                                    │
Week 8:     ┌─── GPT (decoder-only, generation, scaling)
            ├─── BERT (encoder-only, understanding)
            ├─── RWKV / MAMBA (efficient alternatives)
            └─── KAN (rethinking the MLP block)
                                    │
Week 9:     ┌─── Cross-Attention (fuse two modalities)
            ├─── KV Cache (efficient generation)
            ├─── LLM Pipeline (pre-train → SFT → RLHF/DPO)
            ├─── CLIP → LLaVA → Flamingo (vision + language)
            └─── RT-2 / OpenVLA (vision + language + action)
```

The story: **build the Transformer** (Week 7) $\to$ **scale to LLMs** (Week 8) $\to$ **align with humans** (Week 9) $\to$ **extend to vision** (Week 9) $\to$ **extend to action** (Week 9).

---
# Part B · In-Class Coding Exercises

Three exercises building on the mechanisms you just saw. Each task has `# TODO` blanks for you to fill in. A collapsed **Reference solution** cell follows each task — try the task yourself first, then compare.

### Task B1 · Cross-Attention with a padding mask

Section 1 showed cross-attention on two perfectly-sized sequences. In practice, the *context* sequence often contains padded positions that should be ignored. Your job: implement cross-attention with an optional padding mask over the context.

**Shape convention:**

| Tensor | Shape | Meaning |
|---|---|---|
| `x_query` | `(B, T_q, d_model)` | query sequence |
| `x_context` | `(B, T_c, d_model)` | context sequence |
| `context_mask` | `(B, T_c)` or `None` | `1` = valid token, `0` = padding |
| output | `(B, T_q, d_model)` | attended values |

Mask broadcast hint: reshape `context_mask` to `(B, 1, 1, T_c)` so it broadcasts over heads and query positions.

In [ ]:
# ── Task B1: Cross-attention with padding mask (fill in the TODOs) ──────
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MaskedCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_query, x_context, context_mask=None):
        B, T_q, _ = x_query.size()
        _, T_c, _ = x_context.size()
        H, d_k = self.num_heads, self.d_k

        # TODO 1 — project and reshape Q, K, V to (B, H, T_*, d_k)
        Q = None  # <-- replace
        K = None  # <-- replace
        V = None  # <-- replace

        # TODO 2 — scaled dot-product scores, shape (B, H, T_q, T_c)
        scores = None  # <-- replace

        # TODO 3 — apply context_mask: set padded positions to -inf BEFORE softmax.
        #          Hint: mask has shape (B, T_c). Reshape to (B, 1, 1, T_c) for broadcast.
        if context_mask is not None:
            pass  # <-- replace

        # TODO 4 — softmax + weighted sum of V, then concat heads and project out
        attn_weights = None  # <-- replace
        out = None  # <-- replace
        out = out.transpose(1, 2).contiguous().view(B, T_q, -1)
        return self.W_o(out), attn_weights

# ── Quick sanity check (should run once you complete the TODOs) ──
torch.manual_seed(0)
B, T_q, T_c, d, H = 2, 3, 5, 16, 4
x_q = torch.randn(B, T_q, d)
x_c = torch.randn(B, T_c, d)
mask = torch.tensor([[1, 1, 1, 0, 0], [1, 1, 1, 1, 0]])  # rows have 3 and 4 valid context tokens

mca = MaskedCrossAttention(d_model=d, num_heads=H)
# Uncomment once your implementation is complete:
# out, attn = mca(x_q, x_c, context_mask=mask)
# print("output shape:", out.shape)                # expect (2, 3, 16)
# print("attention over padded positions:", attn[:, :, :, 3:].abs().max().item())  # expect ~0

<details>
<summary><b>▸ Reference solution · Task B1</b> (click to expand)</summary>

```python
def forward(self, x_query, x_context, context_mask=None):
    B, T_q, _ = x_query.size()
    _, T_c, _ = x_context.size()
    H, d_k = self.num_heads, self.d_k

    Q = self.W_q(x_query).view(B, T_q, H, d_k).transpose(1, 2)    # (B, H, T_q, d_k)
    K = self.W_k(x_context).view(B, T_c, H, d_k).transpose(1, 2)  # (B, H, T_c, d_k)
    V = self.W_v(x_context).view(B, T_c, H, d_k).transpose(1, 2)  # (B, H, T_c, d_k)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, T_q, T_c)

    if context_mask is not None:
        mask = context_mask.unsqueeze(1).unsqueeze(1)              # (B, 1, 1, T_c)
        scores = scores.masked_fill(mask == 0, float("-inf"))

    attn_weights = F.softmax(scores, dim=-1)                        # (B, H, T_q, T_c)
    out = torch.matmul(attn_weights, V)                             # (B, H, T_q, d_k)
    out = out.transpose(1, 2).contiguous().view(B, T_q, -1)
    return self.W_o(out), attn_weights
```

**Key points:**
- Mask must be applied **before softmax**, not after. Setting masked scores to `-inf` guarantees that `softmax` assigns them exactly zero weight; zeroing out the softmax output afterwards would leave the remaining weights un-normalised.
- The reshape `(B, T_c) → (B, 1, 1, T_c)` lets the mask broadcast across `H` heads and `T_q` query positions without any explicit tiling.
- After a correct implementation, `attn[:, :, :, 3:]` for the first batch row (which has only 3 valid context tokens) should be exactly zero — a useful invariant to check during development.
</details>

### Task B2 · CLIP contrastive loss from scratch

CLIP trains a vision encoder and a text encoder so that matching image–text pairs have high cosine similarity while mismatched pairs have low similarity. Given an L2-normalised batch of image embeddings $I \in \mathbb{R}^{N\times D}$ and text embeddings $T \in \mathbb{R}^{N\times D}$, where row $i$ of $I$ and row $i$ of $T$ are a positive pair, implement the symmetric InfoNCE loss:

$$\mathcal{L}_{\text{CLIP}} = \tfrac{1}{2}\left(\text{CE}(I T^\top/\tau,\, \text{diag}) + \text{CE}(T I^\top/\tau,\, \text{diag})\right)$$

where $\text{diag}$ means "the $i$-th row should classify the $i$-th column as the positive" — i.e., the label for row $i$ is just $i$.

In [ ]:
# ── Task B2: CLIP InfoNCE loss (fill in the TODOs) ───────────────────

def clip_infonce_loss(image_embeds, text_embeds, temperature=0.07):
    """
    Args:
        image_embeds: (N, D) L2-normalised image embeddings
        text_embeds:  (N, D) L2-normalised text embeddings
        temperature:  scalar τ
    Returns:
        scalar symmetric contrastive loss.
    """
    # TODO 1 — similarity matrix (N, N): images on rows, texts on columns, scaled by 1/τ
    logits = None  # <-- replace

    # TODO 2 — labels: the i-th image pairs with the i-th text. Shape (N,), dtype long.
    labels = None  # <-- replace

    # TODO 3 — image-to-text cross entropy (treat each row as a classifier over texts)
    loss_i2t = None  # <-- replace

    # TODO 4 — text-to-image cross entropy (treat each column as a classifier over images)
    #          Hint: transpose the logits.
    loss_t2i = None  # <-- replace

    return (loss_i2t + loss_t2i) / 2

# ── Sanity check: perfectly aligned pairs should give a near-zero loss ──
torch.manual_seed(1)
N, D = 4, 8
I = F.normalize(torch.randn(N, D), dim=-1)
T = I.clone()  # perfect alignment: same vectors for image and text
# Uncomment once complete:
# print("aligned loss (should be small):", clip_infonce_loss(I, T, temperature=0.07).item())
# T2 = F.normalize(torch.randn(N, D), dim=-1)
# print("random loss (should be ~log N):", clip_infonce_loss(I, T2, temperature=0.07).item())

<details>
<summary><b>▸ Reference solution · Task B2</b> (click to expand)</summary>

```python
def clip_infonce_loss(image_embeds, text_embeds, temperature=0.07):
    logits = torch.matmul(image_embeds, text_embeds.T) / temperature   # (N, N)
    labels = torch.arange(len(image_embeds), device=image_embeds.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2
```

**Key points:**
- Treating each row as an $N$-way classifier **is** InfoNCE. Every row's softmax denominator sums over the whole batch, so negatives are the other $N-1$ in-batch examples.
- The temperature $\tau$ controls how sharply the softmax penalises mismatched pairs. Small $\tau$ → harder decision boundary; large $\tau$ → softer. In CLIP, $\tau$ itself is *learned*.
- The symmetric sum (i2t + t2i) is important: dropping one direction would let the model minimise loss by, say, mapping all images onto the same point — a trivial solution that only the t2i term would penalise.
- Sanity checks: identical image/text embeddings should give loss $\to 0$ (modulo temperature); fully random pairs give loss $\approx \log N$ (chance level).
</details>

### Task B3 · KV cache append step

You already saw the cached attention module in §2.3. Your task here is to isolate the **cache update step** and make sure you understand the shape bookkeeping. Given an existing cache `(K_cache, V_cache)` each of shape `(B, t-1, d)`, and a new token embedding `x_new` of shape `(B, 1, d)`, implement a function that returns:

1. the attention output for the new token (shape `(B, 1, d)`), and
2. the updated cache (shape `(B, t, d)` each).

Use pre-built `W_q`, `W_k`, `W_v` matrices (given as function arguments) — no `nn.Module` needed.

In [ ]:
# ── Task B3: KV cache append step (fill in the TODOs) ─────────────────

def kv_cache_step(x_new, W_q, W_k, W_v, kv_cache=None):
    """
    One autoregressive step with a KV cache.
    Args:
        x_new:    (B, 1, d) -- embedding of the new token only
        W_q, W_k, W_v: (d, d) projection matrices
        kv_cache: tuple (K_cache, V_cache) each (B, t-1, d), or None for the first token
    Returns:
        out:           (B, 1, d)  attention output for the new token
        updated_cache: (K, V)     each (B, t, d)
    """
    d = x_new.size(-1)

    # TODO 1 — project the new token to q, k_new, v_new (shape (B, 1, d) each)
    q = None      # <-- replace
    k_new = None  # <-- replace
    v_new = None  # <-- replace

    # TODO 2 — append k_new, v_new to the cache along the time axis.
    #          If kv_cache is None (first step), K = k_new, V = v_new.
    if kv_cache is not None:
        K = None  # <-- replace
        V = None  # <-- replace
    else:
        K, V = k_new, v_new

    # TODO 3 — attention from q (length 1) over all cached keys/values.
    #          No mask needed: everything in cache is already a past token.
    scores = None       # (B, 1, t)
    attn = None         # softmax over last dim
    out = None          # (B, 1, d)

    return out, (K, V)

# ── Sanity check: streaming K,V should match running attention once at the end ──
torch.manual_seed(2)
B, T, d = 2, 5, 8
x = torch.randn(B, T, d)
W_q_, W_k_, W_v_ = torch.randn(d, d), torch.randn(d, d), torch.randn(d, d)

# Uncomment once your implementation is complete to verify streaming:
# cache = None
# outs = []
# for t in range(T):
#     o, cache = kv_cache_step(x[:, t:t+1], W_q_, W_k_, W_v_, cache)
#     outs.append(o)
# streamed = torch.cat(outs, dim=1)  # (B, T, d)
# print("streamed shape:", streamed.shape)  # expect (2, 5, 8)
# print("cache K shape:", cache[0].shape)   # expect (2, 5, 8)

<details>
<summary><b>▸ Reference solution · Task B3</b> (click to expand)</summary>

```python
def kv_cache_step(x_new, W_q, W_k, W_v, kv_cache=None):
    d = x_new.size(-1)
    q     = x_new @ W_q             # (B, 1, d)
    k_new = x_new @ W_k             # (B, 1, d)
    v_new = x_new @ W_v             # (B, 1, d)

    if kv_cache is not None:
        K = torch.cat([kv_cache[0], k_new], dim=1)   # (B, t, d)
        V = torch.cat([kv_cache[1], v_new], dim=1)   # (B, t, d)
    else:
        K, V = k_new, v_new

    scores = torch.matmul(q, K.transpose(-1, -2)) / math.sqrt(d)   # (B, 1, t)
    attn   = F.softmax(scores, dim=-1)
    out    = torch.matmul(attn, V)                                 # (B, 1, d)
    return out, (K, V)
```

**Key points:**
- The **whole point** of a KV cache: we only compute `W_k @ x_new` and `W_v @ x_new` for the single new token. The cache holds the already-computed keys and values for all *earlier* tokens — a classic compute-for-memory trade.
- `torch.cat(..., dim=1)` appends along the time axis. This grows the cache by exactly one step per call.
- No causal mask is needed here because every entry in the cache corresponds to a *past* token by construction. Causal masking is only needed when you compute attention over a full sequence at once (e.g., during prefill or during training).
- Memory cost: after $T$ steps the cache holds $2 \cdot B \cdot T \cdot d$ elements per layer (the factor of 2 is for $K$ and $V$). For long contexts this dominates GPU memory, which is why techniques like PagedAttention and MQA/GQA exist.
</details>

---
# Part C · Exam-Style Questions

Three medium-to-hard short-answer questions covering the core content from this tutorial. Try each question on paper first. Answer sketches follow in collapsed cells — expand only after attempting.

### Q1 — Cross-attention in three VLM paradigms *(connection: self-attention, Week 7)*

Section 4 contrasted three VLM paradigms (CLIP, LLaVA, Flamingo) that integrate vision and language in very different ways.

**(a)** State the shapes of the attention weight matrix for (i) self-attention on a sequence of length $T$ and (ii) cross-attention with query length $T_q$ and context length $T_c$. Give one concrete multimodal example where $T_q \ll T_c$.

**(b)** LLaVA uses **no cross-attention at all**: it projects CLIP image features into the LLM's embedding space through a single MLP, concatenates the resulting image tokens with the text tokens, and lets standard self-attention handle the interaction. Explain *mechanistically* why this "projection + concatenation" approach works, and describe the role self-attention plays on the mixed sequence.

**(c)** Flamingo adds gated cross-attention layers between the visual features and the LLM. The gate is initialised to **zero** and only gradually opens during training. What problem does the zero-initialised gate solve, and what would go wrong if you initialised it to 1?

<details><summary><b>▸ Answer sketch — Q1</b></summary>

**(a)** Self-attention: attention weight matrix is $(T, T)$ — square. Cross-attention: $(T_q, T_c)$ — generally *not* square. A concrete example with $T_q \ll T_c$: in an image-captioning decoder, a single decoder token (query, $T_q = 1$) attends over ~256 image patches (context, $T_c = 256$). Another example: a short user text prompt ($T_q \approx 20$) attending to a long retrieved document ($T_c \approx 2048$) in a retrieval-augmented generation setup.

**(b)** The projection MLP maps each image patch embedding from CLIP's vision space into the *same vector space* the LLM uses for its text tokens. Once the two modalities live in the same space, the LLM's existing self-attention can freely compute dot products between image patches and text tokens — it does not care whether a token came from an image or a word, only whether the query–key inner product is high. Self-attention on the mixed sequence plays a **fusion** role: text tokens that are "looking for" visual information end up with high attention weight on the relevant image patches, and vice versa. No new mechanism is needed — this is the reason LLaVA's architecture is minimally invasive, and a big reason it is popular.

**(c)** The zero-gated cross-attention layer outputs exactly zero at the start of training, so the pre-trained LLM's behavior is **initially unchanged** — adding the gate is mathematically a no-op. As training progresses, the gate opens and visual information starts to flow in, but only once the new cross-attention weights have begun to learn something useful. If the gate were initialised to 1, raw (and initially random) visual information would flood the pre-trained LLM from step 0, **overwhelming** its carefully learned representations and destabilising or erasing them. Zero initialisation preserves the pre-trained behaviour as a **warm start** and lets the model decide for itself how much visual information to integrate. (Same trick is used in LoRA with zero initialisation of $B$, and in ControlNet's zero convolutions.)
</details>

### Q2 — KV cache economics *(connection: efficient inference)*

Consider a decoder-only Transformer with $d_{\text{model}} = 4096$, $L = 32$ layers, and context length $T = 8192$ tokens. Assume $K$ and $V$ are stored per layer in `fp16` (2 bytes per scalar), with a single attention head for simplicity.

**(a)** Derive an expression for the KV cache memory (in bytes) **per sequence** after all $T$ tokens have been generated. Plug in the numbers and express the result in gigabytes.

**(b)** The prefill phase (processing the prompt) is often described as **compute-bound**, while the decode phase (generating one token at a time) is **memory-bandwidth-bound**. Explain why. Your answer should reference (i) how much arithmetic the two phases perform per token and (ii) how much data they need to move from HBM.

**(c)** A classmate says: "KV cache gives us an $O(T)$ speedup for the whole decoding process compared to recomputing attention from scratch at every step." State whether this is tight, loose, or wrong, and give a precise complexity argument for **one generated token at step $t$**, with and without KV cache.

<details><summary><b>▸ Answer sketch — Q2</b></summary>

**(a)** Per layer we store both $K$ and $V$, each of shape $(T, d_{\text{model}})$. Total elements per sequence $= 2 \cdot L \cdot T \cdot d_{\text{model}} = 2 \cdot 32 \cdot 8192 \cdot 4096 \approx 2.15 \times 10^9$ elements. At 2 bytes each (fp16): $\approx 4.3 \times 10^9$ bytes $\approx$ **4.0 GiB per sequence**. This is why serving a single long-context request already consumes multiple gigabytes of HBM, and why batching long-context requests is so expensive.

**(b)** In **prefill**, all $T$ tokens are processed in one forward pass: self-attention does $O(T^2 d)$ arithmetic and the FFN does $O(T d^2)$, both in large matmuls that saturate tensor cores. In **decode**, we generate one token at a time: the attention computation for that token is $O(T d)$ (query over the length-$T$ cache), which is essentially a skinny matrix–vector product; it requires *reading the entire KV cache ($O(T d)$ bytes) from HBM* for barely any arithmetic. Arithmetic intensity (FLOPs per byte) drops by orders of magnitude, so the bottleneck shifts from compute to memory bandwidth. The literature captures this with the roofline model: prefill sits in the compute region, decode sits in the memory-bound region. Techniques like PagedAttention, FlashDecoding, and MQA/GQA exist precisely to reduce the KV-bytes-per-token that need to be streamed during decode.

**(c)** The $O(T)$ statement is **loose — actually quadratic savings**. Without KV cache, generating token $t$ requires re-running self-attention over the whole prefix of length $t$, so **one step costs $O(t^2 d)$**. Summed over all $T$ generated tokens the total cost is $\sum_{t=1}^{T} O(t^2 d) = O(T^3 d)$. With KV cache, generating token $t$ only requires computing $W_q, W_k, W_v$ on the new token ($O(d^2)$) and attention over the length-$t$ cache ($O(t d)$), so **one step costs $O(t d + d^2)$** and the total over all $T$ steps is $O(T^2 d + T d^2)$. The ratio is $T$× at best and $\Theta(T)$ per step — so "$O(T)$ speedup" is right *per generated token*, but cumulative it turns an $O(T^3)$ cost into $O(T^2)$.
</details>

### Q3 — From RLHF to DPO: the implicit reward *(connection: Week 10 bridge)*

Standard RLHF trains an explicit reward model $r_\phi(x, y)$ using the Bradley–Terry preference loss, then optimises the policy with PPO against that reward (plus a KL penalty to a frozen reference $\pi_{\text{ref}}$). DPO skips the reward model and optimises

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)}\!\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right].$$

**(a)** Starting from the KL-constrained RL objective $\max_{\pi} \mathbb{E}_\pi[r(x,y)] - \beta \, \mathrm{KL}(\pi \| \pi_{\text{ref}})$, derive (or sketch clearly) the **closed-form optimal policy** $\pi^\star(y|x)$ and solve for the reward $r$ as a function of $\pi^\star$ and $\pi_{\text{ref}}$. Use this to explain *what* DPO implicitly treats as the reward.

**(b)** Why is the reference model $\pi_{\text{ref}}$ essential? Specifically, what would happen if you replaced $\pi_{\text{ref}}$ with a uniform distribution over tokens?

**(c)** A known failure mode of DPO is that the "loser" loss term can push $\pi_\theta(y_l|x)$ toward zero for every prompt, which collapses entropy and hurts the model's diversity. Using the form of the loss, explain *why* DPO tends toward this collapse, and name one mitigation from the recent literature (e.g. IPO, KTO, length-normalised DPO).

<details><summary><b>▸ Answer sketch — Q3</b></summary>

**(a)** The KL-constrained objective is a Lagrangian whose optimum over $\pi$ is the Gibbs distribution
$$\pi^\star(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x) \exp\!\left(\tfrac{1}{\beta} r(x,y)\right), \qquad Z(x) = \sum_y \pi_{\text{ref}}(y|x) \exp(r(x,y)/\beta).$$
Solving for $r$:
$$r(x,y) = \beta \log \frac{\pi^\star(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x).$$
Plugging this reward into the Bradley–Terry preference model $P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$ makes the $\log Z(x)$ term **cancel** because it is the same function of $x$ for both completions. What remains is exactly the DPO loss, where $\pi_\theta$ plays the role of $\pi^\star$. So DPO implicitly treats the **log-ratio** $\beta \log (\pi_\theta / \pi_{\text{ref}})$ as the reward — any policy that prefers $y_w$ over $y_l$ *relative to the reference* is "high reward" by DPO's standards, with no explicit reward model trained.

**(b)** $\pi_{\text{ref}}$ is the **anchor** that prevents runaway drift from the SFT model and keeps DPO well-posed. Concretely, without $\pi_{\text{ref}}$ the log-ratio loses its grounding: maximising $\log \pi_\theta(y_w)$ and minimising $\log \pi_\theta(y_l)$ in absolute terms is an unconstrained objective, and the model will happily collapse the entire chosen-response probability mass onto a few high-likelihood tokens. Replacing $\pi_{\text{ref}}$ with uniform is equivalent to removing the reference entirely (since $\log (\pi_\theta / U) = \log \pi_\theta + \text{const}$), so the KL constraint in the original objective is lost — DPO becomes a raw likelihood-up-likelihood-down objective and loses all of its theoretical grounding.

**(c)** The loss $-\log \sigma(\beta \Delta)$ with $\Delta = \log(\pi_\theta(y_w)/\pi_{\text{ref}}(y_w)) - \log(\pi_\theta(y_l)/\pi_{\text{ref}}(y_l))$ depends only on the *margin*, not on the absolute likelihoods. The easiest way to increase $\Delta$ is to **drive the loser term $\pi_\theta(y_l|x)$ toward zero** — this is a one-sided optimisation target that requires no trade-off, whereas pushing $\pi_\theta(y_w|x)$ up bumps into the softmax normaliser. Across many preference pairs and many prompts, the model effectively learns to assign near-zero probability to entire families of "slightly worse" continuations, which collapses sampling entropy and reduces output diversity. Mitigations include **IPO** (Azar et al. 2023, replaces the sigmoid with a squared-error loss that penalises over-confident margins), **KTO** (Ethayarajh et al. 2024, uses a prospect-theoretic loss with asymmetric penalties that treats positives and negatives symmetrically in the absolute space), and **length-normalised DPO** variants that explicitly bound the margin per token.
</details>